In [ ]:
import kagglehub
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(path + "/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

for c in df.columns:
  df[c] = df[c].fillna(df[c].mean())

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")
#Imbalanced Use F1/Precision/Recall + StratifiedKFold

In [ ]:
# Task 1: Write your code here:
import numpy as np
X = df.drop('Target', axis=1)
y = df['Target']
X = np.array(X)
y = np.array(y)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Task 2,3,4,5: Write your code here:

def binary_cross_entropy(y, y_hat):
  epsilon = 1e-15  # Very small number to prevent log(0)
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon) # np.clip(value, min, max)

  loss = -1/len(y) * np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
  return loss

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for logistic regression results for each fold
lr_precision = []
lr_recall = []
lr_f1 = []
lr_accuracy = []
lr_loss = []

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y[train_index], y[test_index]

  # 2. Train & Validate CatBoostClassifier
  print(f"Training CatBoostClassifier...")
  model.fit(X_train, y_train) # train

  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred, zero_division=0)
  recall = recall_score(y_test, y_pred, zero_division=0)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  lr_loss.append(binary_cross_entropy(y_test, y_pred))
  lr_precision.append(precision)
  lr_recall.append(recall)
  lr_f1.append(f1)
  lr_accuracy.append(accuracy)
print("DONE!")


In [ ]:
lr_losses = np.array(lr_losses)
lr_precision = np.array(lr_precision)
lr_recall = np.array(lr_recall)
lr_f1 = np.array(lr_f1)

print(lr_losses.mean())
print()
print(lr_precision.mean())
print(lr_recall.mean())
print(lr_f1.mean())

In [ ]:
# Task 1: Write your code here:


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: